# Linear Regression as MLP in PyTorch

In [1]:
#import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [1]:
torch.manual_seed(0)
x = torch.linspace(1, 5, 20)
y = 2*x + 1 + 5*torch.rand(len(x))     # true line 2x + 1 plus noise with mean 2.5
plt.scatter(x, y)

## 1. Dataset Class

In [67]:
class MyData(Dataset):
    def __init__(self, x, y):
        super().__init__()
        self.x = x.view(-1, 1)          # tensors of shape (N, 1)
        self.y = y.view(-1, 1)

    def __len__(self):
        return len(self.y)

    # def transform(self, x):
    #     return torch.cat([x**k for k in range(1,12)], dim=-1)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


data = MyData(x, y)   # nn layers expect shape (batch, features)
print(data[2])

(tensor([1.4211]), tensor([4.2845]))


## 2. Data Loader Class
* `Dataset`: how to get the $i$-th example;
* `DataLoader`: groups examples into batches and **shuffles** them every epoch (not needed for test);
The number of batches is $\lceil N/B\rceil$, if `drop_last=False`, otherwise $\lfloor N/B\rfloor$.

In [68]:
# shuffle=True: a new random split into batches every epoch
data_loader = DataLoader(data, batch_size=5, shuffle=True)
for x_batch, y_batch in data_loader:
    print(x_batch.shape, y_batch.shape)
    break

torch.Size([5, 1]) torch.Size([5, 1])


## 3. Model Class

In [69]:
torch.manual_seed(0)
model = nn.Linear(1, 1)
print(list(model.parameters()))       # weight w and bias b, created with requires_grad=True

[Parameter containing:
tensor([[-0.0075]], requires_grad=True), Parameter containing:
tensor([0.5364], requires_grad=True)]


In [70]:
class MyLinearReg(nn.Module):
    def __init__(self, input_size=1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 1)     # W1 x + b1

    def forward(self, x_batch):
        o = self.fc1(x_batch)
        return o

In [71]:
model = MyLinearReg()
print(model)

MyLinearReg(
  (fc1): Linear(in_features=1, out_features=1, bias=True)
)


## 4. Mini-batches, `nn.Linear` and an optimizer

Instead of all $N$ points, each update uses a random **mini-batch** of $B$ points. One pass over the data is an **epoch**; one update is an **iteration**, so an epoch has $\lceil N/B\rceil$ iterations (if `drop_last=False`).

PyTorch splits this into three pieces:
* `Dataset`: how to get the $i$-th example;
* `DataLoader`: groups examples into batches and **shuffles** them every epoch;
* `optimizer`: performs the update `w -= lr * w.grad` (or a smarter one) for all parameters.

In [72]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

The mini-batch training loop (200 epochs).
Per batch: prediction, loss, `backward`, `optimizer.step()`, `optimizer.zero_grad()`. Store `loss.item()` in `hist`.

In [2]:
hist = []
for epoch in range(200):
    for x_b, y_b in data_loader:
        y_pred = model(x_b)            # model(x) calls model.forward(x)
        loss = criterion(y_pred, y_b)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        hist.append(loss.item())       # .item(): a plain number, no graph kept

        # visualisation:
    if (epoch + 1) % 2 == 0:
        clear_output(True)
        plt.figure(figsize=(8, 6))
        plt.scatter(x, y, label="data")
        plt.plot(x, model(data.x).detach().numpy(), color="orange", linewidth=3, label="predictions")
        plt.title(f"epoch {epoch+1}, loss {loss.item():.3f}")
        plt.legend()
        plt.show()

plt.plot(hist)
plt.xlabel('iteration'); plt.ylabel('batch loss')
print(model.fc1.weight.item(), model.fc1.bias.item())

`nn.Linear(1, 1)` is exactly $wx+b$: linear regression is a neural network with no hidden layer.

## 4b. More data, a train/test split, and TensorBoard

With 20 points we can only watch the training loss. Now we generate **500** points from the same law, keep 20% aside as a **test set** the model never trains on, and log everything to **TensorBoard**:

* **Scalars**: the train and test loss per epoch in one chart (two learning curves), plus $w$ and $b$;
* **Images, tag `fit`**: the fitted line every 2 epochs, one image per step (drag the step slider);
* **Images, tag `fit_movie`**: the same frames as one animation that **plays by itself** inside TensorBoard (an animated GIF, the format `add_video` uses).

The noise $5\cdot U(0,1)$ has variance $25/12\approx 2.08$, so even the true mean line $y = 2x + 3.5$ has an **expected** MSE of about 2.08. On a finite test set the measured value scatters around that: with 100 points it can land a bit above or below.

In [3]:
from torch.utils.data import random_split

torch.manual_seed(0)
N = 500
x_big = 1 + 4 * torch.rand(N)                       # x uniform on [1, 5]
y_big = 2 * x_big + 1 + 5 * torch.rand(N)           # same law as before
full = MyData(x_big, y_big)

# 80% train / 20% test; the generator makes the split reproducible
train_set, test_set = random_split(full, [400, 100], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=100, shuffle=False)   # no need to shuffle for evaluation

# Subset objects keep the indices of their points, handy for plotting
x_tr, y_tr = full.x[train_set.indices], full.y[train_set.indices]
x_te, y_te = full.x[test_set.indices], full.y[test_set.indices]
plt.scatter(x_tr, y_tr, s=10, label=f'train ({len(train_set)})')
plt.scatter(x_te, y_te, s=10, label=f'test ({len(test_set)})')
plt.legend(); plt.show()

Start TensorBoard **before** training, so you can watch the curves appear (refresh, or turn on auto-reload in its settings). In Colab or Jupyter the two magics below open it inside the notebook; from a terminal, run `tensorboard --logdir runs` and open http://localhost:6006.

In [26]:
from torch.utils.tensorboard import SummaryWriter

In [80]:
#%rm -r runs

In [4]:
%load_ext tensorboard
%tensorboard --logdir runs --samples_per_plugin images=500

In [83]:
def evaluate(model, loader, criterion):
    """Mean loss over a whole dataset, without building a graph."""
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x_b, y_b in loader:
            total += criterion(model(x_b), y_b).item() * len(x_b)   # weight by batch size
            n += len(x_b)
    model.train()
    return total / n


x_grid = torch.linspace(0.8, 5.2, 50).view(-1, 1)

def frame(model, epoch):
    """Figure of the data and the current fitted line; fixed axes so the movie doesn't jump."""
    with torch.no_grad():
        y_grid = model(x_grid)
    fig = plt.figure(figsize=(5, 3.75), dpi=80)
    plt.scatter(x_tr, y_tr, s=6, alpha=.4, label='train')
    plt.scatter(x_te, y_te, s=6, alpha=.6, label='test')
    plt.plot(x_grid, y_grid, color='black', lw=2.5, label='model')
    plt.plot(x_grid, 2 * x_grid + 3.5, 'g--', lw=1, label='true mean 2x + 3.5')
    plt.xlim(0.8, 5.2); plt.ylim(0, 18); plt.legend(loc='upper left', fontsize=7)
    plt.title(f'epoch {epoch}')
    return fig

In [85]:
torch.manual_seed(0)
model = MyLinearReg()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0002)
writer = SummaryWriter(log_dir='runs/linreg_sgd_lr0.02')   # one folder per experiment

n_epochs = 100
train_hist, test_hist = [], []
for epoch in range(n_epochs):
    # ---- one epoch of training (the loop from Q4) ----
    running, n = 0.0, 0
    for x_b, y_b in train_loader:
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        running += loss.item() * len(x_b)
        n += len(x_b)
    train_loss = running / n                       # mean loss over this epoch's batches
    test_loss = evaluate(model, test_loader, criterion)
    train_hist.append(train_loss); test_hist.append(test_loss)

    # ---- logging ----
    writer.add_scalars('loss', {'train': train_loss, 'test': test_loss}, epoch)   # two curves, one chart
    writer.add_scalar('params/w', model.fc1.weight.item(), epoch)
    writer.add_scalar('params/b', model.fc1.bias.item(), epoch)
    if epoch % 2 == 0 or epoch == n_epochs - 1:
        writer.add_figure('fit', frame(model, epoch), global_step=epoch)   # Images tab: one frame per step

writer.close()

print(f'w = {model.fc1.weight.item():.3f}, b = {model.fc1.bias.item():.3f}')
print(f'final MSE: train {train_hist[-1]:.3f}, test {test_hist[-1]:.3f}  (expected noise floor {25/12:.2f})')

w = 2.566, b = 1.579
final MSE: train 2.554, test 3.257  (expected noise floor 2.08)


# **PROMLEMS**
1. Modify the previouse code 1--4b for linear regression to polinomial regression (of power 2), i.e., $a(x) = w_0 + w_1x + w_2 x^2$

2. Train the model

## 5. A two-layer perceptron

Now a target that no line can fit: $y = |x|$. We change **only the model**; the loop from Q4 is reused.

$$a(x) = W_2\,\sigma(W_1 x + b_1) + b_2$$

In [ ]:
x2 = torch.linspace(-1, 1, 100).view(-1, 1)
y2 = torch.abs(x2)
plt.plot(x2, y2)

data2 = MyData(x2, y2)
data_loader2 = DataLoader(data2, batch_size=10, shuffle=True)

In [ ]:
class MyModel_MLP(nn.Module):
    def __init__(self, hidden_size, activation=nn.Sigmoid()):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_size)     # W1 x + b1
        self.activ = activation                  # sigma, applied elementwise
        self.fc2 = nn.Linear(hidden_size, 1)     # W2 h + b2

    def forward(self, x_batch):
        o = self.fc1(x_batch)
        o = self.activ(o)
        o = self.fc2(o)                          # keep the output of the last layer
        return o

In [ ]:
def train(model, loader, epochs=100, lr=0.01):
    # the same loop as in Q4, with Adam instead of plain SGD
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    hist = []
    for epoch in range(epochs):
        for x_b, y_b in loader:
            loss = criterion(model(x_b), y_b)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            hist.append(loss.item())
    return hist


def show(model, title):
    with torch.no_grad():                # no graph needed for plotting
        y_pred = model(x2)
    plt.scatter(x2, y2, s=8, label='|x|')
    plt.plot(x2, y_pred, color='orange', linewidth=3, label='model')
    plt.title(title); plt.legend(); plt.show()

### Train three models and compare the fits:
1. `nn.Linear(1, 1)` (no hidden layer);
2. `MyModel(5)` with the sigmoid;
3. `MyModel(8, nn.ReLU())`.

In [ ]:
torch.manual_seed(0)
for name, m in [('linear', nn.Linear(1, 1)),
                ('MLP, 5 sigmoid units', MyModel_MLP(5)),
                ('MLP, 8 ReLU units', MyModel_MLP(8, nn.ReLU()))]:
    hist = train(m, data_loader2, epochs=200, lr=0.01)
    show(m, f'{name}: final batch loss {hist[-1]:.4f}')


### Exact representation.
Show that $|x| = \mathrm{ReLU}(x) + \mathrm{ReLU}(-x)$, and write down $W_1, b_1, W_2, b_2$ of a network with **2** ReLU units that equals $|x|$ exactly.
Then train `MyModel(2, nn.ReLU())` from several random seeds. Does training always find this solution? What goes wrong when it does not? *(Hint: look at units whose pre-activation $W_1x+b_1$ is negative for every $x\in[-1,1]$.)*

In [ ]:
# W1 = [[1], [-1]], b1 = [0, 0], W2 = [[1, 1]], b2 = [0]
exact = MyModel_MLP(2, nn.ReLU())
with torch.no_grad():
    exact.fc1.weight.copy_(torch.tensor([[1.], [-1.]]))
    exact.fc1.bias.zero_()
    exact.fc2.weight.copy_(torch.tensor([[1., 1.]]))
    exact.fc2.bias.zero_()
show(exact, 'hand-set weights: exact')

for seed in range(4):
    torch.manual_seed(seed)
    m = MyModel_MLP(2, nn.ReLU())
    hist = train(m, data_loader2, epochs=200, lr=0.01)
    show(m, f'seed {seed}: final batch loss {hist[-1]:.4f}')
# Some seeds may give a "dead" unit (ReLU output 0 on the whole interval -> zero gradient),
# and the network is stuck with a single kink.
